# DL200 Exercises -- MLP Design and Training Practices

Work through these exercises after completing the DL200 module. Try each exercise before looking at the solutions at the bottom.

**Topics covered:** MLP architecture design, diagnosing overfitting/underfitting from loss curves, normalization techniques, data leakage identification.

---
## Exercise 1: MLP Architecture Design

You are given a **tabular regression** task with:
- 50 input features
- 10,000 training samples
- Target: continuous value (house price)

Design an MLP architecture by answering:

1. How many hidden layers would you use? Why?
2. How many neurons per hidden layer? What is a reasonable starting point?
3. What activation function for hidden layers? What about the output layer?
4. What loss function?
5. Would you use dropout? If so, what rate?
6. Would you use batch normalization? Where in the network?

In [ ]:
import torch
import torch.nn as nn

# TODO: Implement your MLP design as a nn.Module
# Requirements:
# - input_dim = 50
# - output_dim = 1 (regression)
# - Include your chosen hidden layers, activations, dropout, batch norm

class HousePriceMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: define layers
        pass

    def forward(self, x):
        # TODO: forward pass
        pass

# Print model summary
# model = HousePriceMLP()
# print(model)
# print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## Exercise 2: Diagnosing Overfitting from Loss Curves

The training curves below show different scenarios. For each one, diagnose the problem and suggest a fix.

**Run the cell to see the plots, then answer the questions below.**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Scenario A: Classic overfitting
epochs = np.arange(1, 51)
train_a = 2.0 * np.exp(-0.08 * epochs) + 0.05
val_a = 1.5 * np.exp(-0.05 * epochs) + 0.3 + 0.01 * epochs
axes[0].plot(epochs, train_a, label="Train Loss", linewidth=2)
axes[0].plot(epochs, val_a, label="Val Loss", linewidth=2)
axes[0].set_title("Scenario A")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scenario B: Underfitting
train_b = 2.0 * np.exp(-0.02 * epochs) + 0.8
val_b = 2.0 * np.exp(-0.018 * epochs) + 0.85
axes[1].plot(epochs, train_b, label="Train Loss", linewidth=2)
axes[1].plot(epochs, val_b, label="Val Loss", linewidth=2)
axes[1].set_title("Scenario B")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Scenario C: Good fit
train_c = 2.0 * np.exp(-0.06 * epochs) + 0.15
val_c = 2.0 * np.exp(-0.05 * epochs) + 0.22
axes[2].plot(epochs, train_c, label="Train Loss", linewidth=2)
axes[2].plot(epochs, val_c, label="Val Loss", linewidth=2)
axes[2].set_title("Scenario C")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Loss")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Questions:**

1. **Scenario A:** What is the diagnosis? Name 3 techniques to fix it.
2. **Scenario B:** What is the diagnosis? Name 3 techniques to fix it.
3. **Scenario C:** What does this curve indicate? Is anything wrong?

---
## Exercise 3: Feature Normalization

You have a dataset where features have vastly different scales:

| Feature | Min | Max | Mean | Std |
|---------|-----|-----|------|-----|
| Age | 18 | 90 | 45 | 15 |
| Income | 20000 | 500000 | 65000 | 40000 |
| Num_clicks | 0 | 10000 | 200 | 500 |
| Rating | 1.0 | 5.0 | 3.5 | 0.8 |

1. Why is normalization important for neural networks?
2. Implement both **StandardScaler** (z-score) and **MinMaxScaler** (0-1 scaling) from scratch using PyTorch.
3. Which would you choose here and why?
4. **Critical:** Should you fit the scaler on training data only or all data? Why?

In [ ]:
import torch

# Sample data
torch.manual_seed(42)
X_train = torch.tensor([
    [25, 30000, 50, 4.2],
    [45, 75000, 200, 3.8],
    [60, 120000, 500, 2.5],
    [35, 55000, 100, 4.0],
    [50, 90000, 300, 3.2],
], dtype=torch.float32)

X_test = torch.tensor([
    [30, 40000, 80, 3.9],
    [55, 100000, 400, 2.8],
], dtype=torch.float32)

# TODO: Implement StandardScaler
def standard_scale_fit(X_train):
    """Compute mean and std from training data."""
    pass

def standard_scale_transform(X, mean, std):
    """Apply z-score normalization."""
    pass

# TODO: Implement MinMaxScaler
def minmax_scale_fit(X_train):
    """Compute min and max from training data."""
    pass

def minmax_scale_transform(X, min_val, max_val):
    """Scale to [0, 1] range."""
    pass

---
## Exercise 4: Identifying Data Leakage

Each scenario below contains a form of **data leakage**. Identify the leak and explain how to fix it.

### Scenario A
```python
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Normalize BEFORE splitting
scaler = StandardScaler()
X_normalized = scaler.fit_transform(X_all)  # <-- fit on ALL data

X_train, X_test = train_test_split(X_normalized, test_size=0.2)
# Train model on X_train, evaluate on X_test
```

### Scenario B
```python
# Feature engineering using target info
df['price_category'] = pd.cut(df['price'], bins=3, labels=['low', 'mid', 'high'])
# Then use price_category as a feature to predict... price
```

### Scenario C
```python
# Time-series data: random splitting
X_train, X_test = train_test_split(stock_data, test_size=0.2, random_state=42)
# Training on random mixture of past and future data
```

### Scenario D
```python
# Medical data: patient with multiple visits
# Same patient appears in both train and test splits
X_train, X_test = train_test_split(patient_visits, test_size=0.2)
```

**Your answers for each scenario:**

---
## Exercise 5: Regularization Experiment

Implement a simple experiment to observe the effect of dropout on overfitting. Train the same network architecture with and without dropout on a small synthetic dataset.

1. Create a model with dropout=0.0 and train for 100 epochs
2. Create the same model with dropout=0.5 and train for 100 epochs
3. Plot the training and validation loss for both
4. Which model overfits more?

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# Small synthetic dataset (easy to overfit)
n_train, n_val = 100, 50
X_train = torch.randn(n_train, 20)
y_train = (X_train[:, 0] * 2 + X_train[:, 1] - X_train[:, 2] + torch.randn(n_train) * 0.5)
X_val = torch.randn(n_val, 20)
y_val = (X_val[:, 0] * 2 + X_val[:, 1] - X_val[:, 2] + torch.randn(n_val) * 0.5)

class RegularizationNet(nn.Module):
    def __init__(self, dropout_rate=0.0):
        super().__init__()
        # TODO: Define a model with:
        # Linear(20, 128) -> ReLU -> Dropout -> Linear(128, 64) -> ReLU -> Dropout -> Linear(64, 1)
        pass

    def forward(self, x):
        # TODO
        pass

def train_and_record(model, X_train, y_train, X_val, y_val, epochs=100, lr=0.01):
    # TODO: Train and return lists of train_loss and val_loss per epoch
    pass

# TODO: Train both models and plot results

---
## Exercise 6: Weight Initialization Impact

Compare the effect of different weight initialization strategies on a 10-layer deep network.

1. **Zero initialization**: All weights set to 0
2. **Too-large random**: Weights sampled from N(0, 1)
3. **Xavier/Glorot**: Proper initialization for tanh/sigmoid
4. **He/Kaiming**: Proper initialization for ReLU

For each, pass a random input through the network and print the mean and standard deviation of the output. Which ones produce reasonable outputs?

In [ ]:
import torch
import torch.nn as nn

def make_deep_net(init_method, n_layers=10, dim=256):
    """Create a deep network with the specified initialization."""
    layers = []
    for _ in range(n_layers):
        linear = nn.Linear(dim, dim)

        if init_method == "zeros":
            nn.init.zeros_(linear.weight)
            nn.init.zeros_(linear.bias)
        elif init_method == "large_random":
            nn.init.normal_(linear.weight, std=1.0)
            nn.init.zeros_(linear.bias)
        elif init_method == "xavier":
            # TODO: use xavier_uniform_
            pass
        elif init_method == "kaiming":
            # TODO: use kaiming_normal_ with mode='fan_in', nonlinearity='relu'
            pass

        layers.append(linear)
        layers.append(nn.ReLU())

    return nn.Sequential(*layers)

# TODO: Test each initialization with a random input of shape (32, 256)
# Print the mean and std of the output for each

---
## Exercise 7: Hyperparameter Sensitivity

The learning rate is one of the most important hyperparameters. Train a simple MLP on the synthetic data below with the following learning rates: `[0.0001, 0.001, 0.01, 0.1, 1.0]`.

1. Plot the training loss curves for all learning rates on the same graph
2. Which learning rate converges fastest?
3. Which learning rate fails to converge?
4. What happens with a very large learning rate (1.0)?

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Synthetic data
torch.manual_seed(42)
X = torch.randn(500, 10)
y = (X @ torch.randn(10, 1) + torch.randn(500, 1) * 0.1).squeeze()

learning_rates = [0.0001, 0.001, 0.01, 0.1, 1.0]

# TODO: For each learning rate:
# 1. Create a fresh model: Linear(10, 32) -> ReLU -> Linear(32, 1)
# 2. Train for 50 epochs with MSE loss
# 3. Record the loss at each epoch
# 4. Plot all curves on the same graph

---
---
# Solutions

Expand each section below to see the solution.

<details>
<summary><b>Solution 1: MLP Architecture Design</b></summary>

**Design rationale:**

1. **2-3 hidden layers.** For tabular regression with 10K samples and 50 features, a shallow-to-moderate network is sufficient. Going deeper risks overfitting without benefit.

2. **64-128 neurons per layer** as a starting point, with a "funnel" pattern (e.g., 128 -> 64 -> 32). The first layer should generally be larger than the input dimension.

3. **ReLU for hidden layers** (standard choice), **no activation on output** (regression outputs raw values).

4. **MSE loss** for regression.

5. **Dropout 0.2-0.3.** With 10K samples, some regularization helps but heavy dropout is unnecessary.

6. **BatchNorm before activation** in each hidden layer to stabilize training.

```python
class HousePriceMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(50, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Linear(32, 1),  # No activation for regression
        )

    def forward(self, x):
        return self.net(x)

model = HousePriceMLP()
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
```
</details>

<details>
<summary><b>Solution 2: Diagnosing Overfitting from Loss Curves</b></summary>

**Scenario A: Classic Overfitting**
- **Diagnosis:** Training loss decreases steadily, but validation loss decreases initially then increases. The growing gap between train and val loss indicates the model is memorizing training data.
- **Fixes:** (1) Add dropout, (2) Reduce model capacity (fewer layers/neurons), (3) Add L2 weight decay, (4) Use early stopping, (5) Collect more training data, (6) Add data augmentation.

**Scenario B: Underfitting**
- **Diagnosis:** Both training and validation loss are high and barely decrease. The model lacks capacity to learn the patterns.
- **Fixes:** (1) Increase model capacity (more layers/neurons), (2) Train for more epochs, (3) Reduce regularization (lower dropout, less weight decay), (4) Use a lower learning rate with more epochs, (5) Improve feature engineering.

**Scenario C: Good Fit**
- **Diagnosis:** Both losses decrease and converge to similar values. The small gap between train and val loss is normal and healthy. Nothing is wrong.
- **Note:** Could potentially improve by training a bit longer or with a slightly larger model, but this is already a good result.
</details>

<details>
<summary><b>Solution 3: Feature Normalization</b></summary>

**Why normalization matters:** Without it, features with large magnitudes (like Income) dominate the gradient, making training slow and unstable. Neural networks learn best when inputs are roughly the same scale.

```python
# StandardScaler (z-score normalization)
def standard_scale_fit(X_train):
    mean = X_train.mean(dim=0)
    std = X_train.std(dim=0)
    return mean, std

def standard_scale_transform(X, mean, std):
    return (X - mean) / (std + 1e-8)  # epsilon to avoid division by zero

# MinMaxScaler (0-1 scaling)
def minmax_scale_fit(X_train):
    min_val = X_train.min(dim=0).values
    max_val = X_train.max(dim=0).values
    return min_val, max_val

def minmax_scale_transform(X, min_val, max_val):
    return (X - min_val) / (max_val - min_val + 1e-8)

# Apply
mean, std = standard_scale_fit(X_train)
X_train_scaled = standard_scale_transform(X_train, mean, std)
X_test_scaled = standard_scale_transform(X_test, mean, std)
```

**Which to choose:** StandardScaler is generally preferred for neural networks. It centers data at 0 with unit variance, which works well with weight initialization assumptions.

**Critical:** Always fit the scaler on **training data only**, then use those same statistics to transform validation and test data. Fitting on all data leaks test set information into preprocessing, inflating performance estimates.
</details>

<details>
<summary><b>Solution 4: Identifying Data Leakage</b></summary>

**Scenario A -- Preprocessing Leakage:**
The scaler is fit on ALL data (including test), so the mean/std incorporate test set statistics. **Fix:** Fit scaler on X_train only, then transform X_test with the fitted scaler.
```python
X_train, X_test = train_test_split(X_all, test_size=0.2)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # fit ONLY on train
X_test = scaler.transform(X_test)         # transform test with train stats
```

**Scenario B -- Target Leakage:**
`price_category` is derived directly from the target variable `price`. Using it as a feature gives the model direct access to the answer. **Fix:** Remove any features derived from the target. Only use features that would be available at prediction time.

**Scenario C -- Temporal Leakage:**
Random splitting of time-series data means the model trains on future data to predict the past. **Fix:** Use a chronological split -- train on data before a cutoff date, test on data after:
```python
cutoff = len(stock_data) - int(0.2 * len(stock_data))
X_train, X_test = stock_data[:cutoff], stock_data[cutoff:]
```

**Scenario D -- Group Leakage:**
The same patient appears in both train and test. The model memorizes patient-specific patterns. **Fix:** Use `GroupShuffleSplit` or `GroupKFold` to ensure all visits from the same patient are in the same split:
```python
from sklearn.model_selection import GroupShuffleSplit
splitter = GroupShuffleSplit(test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=patient_ids))
```
</details>

<details>
<summary><b>Solution 5: Regularization Experiment</b></summary>

```python
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)
n_train, n_val = 100, 50
X_train = torch.randn(n_train, 20)
y_train = (X_train[:, 0] * 2 + X_train[:, 1] - X_train[:, 2]
           + torch.randn(n_train) * 0.5)
X_val = torch.randn(n_val, 20)
y_val = (X_val[:, 0] * 2 + X_val[:, 1] - X_val[:, 2]
         + torch.randn(n_val) * 0.5)

class RegularizationNet(nn.Module):
    def __init__(self, dropout_rate=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(20, 128), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze()

def train_and_record(model, X_train, y_train, X_val, y_val, epochs=100, lr=0.01):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    train_losses, val_losses = [], []
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(model(X_train), y_train)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            val_losses.append(loss_fn(model(X_val), y_val).item())
    return train_losses, val_losses

torch.manual_seed(42)
model_no_drop = RegularizationNet(dropout_rate=0.0)
tl1, vl1 = train_and_record(model_no_drop, X_train, y_train, X_val, y_val)

torch.manual_seed(42)
model_dropout = RegularizationNet(dropout_rate=0.5)
tl2, vl2 = train_and_record(model_dropout, X_train, y_train, X_val, y_val)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(tl1, label="Train (no dropout)"); ax1.plot(vl1, label="Val (no dropout)")
ax1.set_title("No Dropout"); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(tl2, label="Train (dropout=0.5)"); ax2.plot(vl2, label="Val (dropout=0.5)")
ax2.set_title("Dropout=0.5"); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
```

The model without dropout will show a larger gap between training and validation loss (overfitting). Dropout regularizes by preventing co-adaptation of neurons.
</details>

<details>
<summary><b>Solution 6: Weight Initialization Impact</b></summary>

```python
import torch
import torch.nn as nn

def make_deep_net(init_method, n_layers=10, dim=256):
    layers = []
    for _ in range(n_layers):
        linear = nn.Linear(dim, dim)
        if init_method == "zeros":
            nn.init.zeros_(linear.weight)
            nn.init.zeros_(linear.bias)
        elif init_method == "large_random":
            nn.init.normal_(linear.weight, std=1.0)
            nn.init.zeros_(linear.bias)
        elif init_method == "xavier":
            nn.init.xavier_uniform_(linear.weight)
            nn.init.zeros_(linear.bias)
        elif init_method == "kaiming":
            nn.init.kaiming_normal_(linear.weight, mode='fan_in', nonlinearity='relu')
            nn.init.zeros_(linear.bias)
        layers.append(linear)
        layers.append(nn.ReLU())
    return nn.Sequential(*layers)

x = torch.randn(32, 256)
for method in ["zeros", "large_random", "xavier", "kaiming"]:
    torch.manual_seed(42)
    net = make_deep_net(method)
    with torch.no_grad():
        out = net(x)
    print(f"{method:15s}: mean={out.mean().item():.6f}, std={out.std().item():.6f}")
```

**Results:**
- **Zeros:** Output is all zeros. Gradients are zero. Model cannot learn -- all neurons are identical (symmetry problem).
- **Large random:** Output has huge std (possibly inf/nan). Gradients explode.
- **Xavier:** Reasonable output, but slightly suboptimal for ReLU (designed for tanh/sigmoid).
- **Kaiming:** Best for ReLU -- maintains variance through layers. This is the recommended initialization for ReLU networks.
</details>

<details>
<summary><b>Solution 7: Hyperparameter Sensitivity</b></summary>

```python
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

X = torch.randn(500, 10)
y = (X @ torch.randn(10, 1) + torch.randn(500, 1) * 0.1).squeeze()

learning_rates = [0.0001, 0.001, 0.01, 0.1, 1.0]
all_losses = {}

for lr in learning_rates:
    torch.manual_seed(42)
    model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 1))
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    losses = []

    for epoch in range(50):
        optimizer.zero_grad()
        pred = model(X).squeeze()
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    all_losses[lr] = losses

fig, ax = plt.subplots(figsize=(10, 6))
for lr, losses in all_losses.items():
    ax.plot(losses, label=f"lr={lr}", linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Learning Rate Comparison")
ax.set_yscale("log")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
```

**Observations:**
- **lr=0.0001:** Very slow convergence (barely moves in 50 epochs).
- **lr=0.001:** Steady convergence, moderate speed.
- **lr=0.01:** Good convergence speed -- often the sweet spot.
- **lr=0.1:** Fast initial convergence but may oscillate.
- **lr=1.0:** Loss explodes or oscillates wildly. The step size is too large, overshooting the minimum each time.
</details>